In [ ]:
import torch
print("GPU:", torch.cuda.is_available())

GPU: True


In [ ]:
!pip install -q transformers datasets scikit-learn pandas matplotlib

In [ ]:
import pandas as pd

train = pd.read_csv("data/combined_data/train.csv")
print(train.head())

                                            sentence     label     id  \
0  WH also a strong day today, looks ready for th...  positive  12863   
1  $MUX - McEwen Mining: The Writing Was On The W...   neutral   8224   
2  user: DOV weekly. Going higher - bot into clos...  positive  13867   
3  Netflix downgraded to underperform at Wells Fargo  negative   9556   
4  `` The implementation of these programs has ha...  negative  41173   

   token_count  char_count  avg_word_len  dep_depth  num_clauses  \
0           14          59         3.538          4            1   
1           27         136         4.778          4            0   
2           22          97         3.944          5            1   
3            7          49         6.143          3            0   
4           27         134         5.167          7            1   

   passive_voice_count  negation_count  hedge_count  num_financial_terms  \
0                    0               0            0                    0   


Import

In [ ]:
# Large-model baseline for financial sentiment classification


import os
import json
import math
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt



Config

In [ ]:

# 1. Configuration
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_ROOT = Path(".")
DATA_DIR = REPO_ROOT / "data" / "combined_data"
OUTPUT_DIR = REPO_ROOT / "Output" / "model_outputs" / "Finetuned_Roberta_large" /"with_Combined_Dataset"
MODEL_DIR = REPO_ROOT / "models" / "finetuned_roberta_large" / "with_Combined_Dataset"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TEST_PATH = DATA_DIR / "test.csv"

MODEL_NAME = "roberta-large"

MAX_LEN = 128
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 4
WARMUP_RATIO = 0.1
GRAD_ACCUM_STEPS = 2   # effective batch size = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS
FP16 = torch.cuda.is_available()



Utilities

In [ ]:
# 2. Utilities

def detect_text_col(df: pd.DataFrame) -> str:
    candidates = ["sentence", "text", "content", "query"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find text column. Available columns: {list(df.columns)}")


def detect_label_col(df: pd.DataFrame) -> str:
    candidates = ["label", "labels", "target", "sentiment"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find label column. Available columns: {list(df.columns)}")


def detect_id_col(df: pd.DataFrame) -> str:
    candidates = ["id", "idx", "index"]
    for c in candidates:
        if c in df.columns:
            return c
    
    return ""


def build_label_maps(train_df: pd.DataFrame, label_col: str) -> Tuple[Dict, Dict]:
    unique_labels = sorted(train_df[label_col].dropna().unique().tolist())

    # If labels are strings, map them to ints in stable order
    if isinstance(unique_labels[0], str):
        # Prefer negative-neutral-positive if present
        canonical = ["negative", "neutral", "positive"]
        if all(lbl in unique_labels for lbl in canonical):
            label2id = {lbl: i for i, lbl in enumerate(canonical)}
        else:
            label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
        id2label = {v: k for k, v in label2id.items()}
    else:
        label2id = {int(lbl): int(lbl) for lbl in unique_labels}
        id2label = {int(lbl): str(int(lbl)) for lbl in unique_labels}

    return label2id, id2label


def encode_labels(df: pd.DataFrame, label_col: str, label2id: Dict) -> pd.DataFrame:
    df = df.copy()
    if df[label_col].dtype == object:
        df[label_col] = df[label_col].map(label2id)
    df[label_col] = df[label_col].astype(int)
    return df


def compute_entropy(probs: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    probs = np.clip(probs, eps, 1.0)
    return -(probs * np.log(probs)).sum(axis=1)


def compute_margin(probs: np.ndarray) -> np.ndarray:
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    return sorted_probs[:, 0] - sorted_probs[:, 1]


def softmax(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }


def save_confusion_matrix(y_true, y_pred, labels, out_path: Path, title: str):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest")
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=labels,
        yticklabels=labels,
        ylabel="True label",
        xlabel="Predicted label",
        title=title,
    )

    thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    fig.tight_layout()
    plt.savefig(out_path, bbox_inches="tight", dpi=200)
    plt.close(fig)


def prepare_hf_dataset(df: pd.DataFrame, text_col: str, label_col: str) -> Dataset:
    keep_cols = [text_col, label_col]
    if "id" in df.columns:
        keep_cols.append("id")
    ds = Dataset.from_pandas(df[keep_cols], preserve_index=False)
    return ds


Load Data

In [ ]:

# 3. Load data
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

text_col = detect_text_col(train_df)
label_col = detect_label_col(train_df)
id_col = detect_id_col(train_df)

if not id_col:
    for df_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        df["id"] = np.arange(len(df))
    id_col = "id"

label2id, id2label = build_label_maps(train_df, label_col)

train_df = encode_labels(train_df, label_col, label2id)
val_df = encode_labels(val_df, label_col, label2id)
test_df = encode_labels(test_df, label_col, label2id)

print("Text column:", text_col)
print("Label column:", label_col)
print("ID column:", id_col)
print("Label mapping:", label2id)
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)



Text column: sentence
Label column: label
ID column: id
Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Train shape: (29040, 17)
Val shape: (6223, 17)
Test shape: (6223, 17)


In [ ]:
print(train_df['label'].value_counts())
print(val_df['label'].value_counts())
print(test_df['label'].value_counts())

label
2    12769
1    10004
0     6267
Name: count, dtype: int64
label
2    2736
1    2144
0    1343
Name: count, dtype: int64
label
2    2737
1    2143
0    1343
Name: count, dtype: int64


In [ ]:
print(train_df['label'].value_counts(normalize=True) * 100)

label
2    43.970386
1    34.449036
0    21.580579
Name: proportion, dtype: float64


Tokenizer

In [ ]:

# 4. Tokenizer + datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(examples):
    return tokenizer(
        examples[text_col],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = prepare_hf_dataset(train_df, text_col, label_col)
val_ds = prepare_hf_dataset(val_df, text_col, label_col)
test_ds = prepare_hf_dataset(test_df, text_col, label_col)

train_ds = train_ds.rename_column(label_col, "labels")
val_ds = val_ds.rename_column(label_col, "labels")
test_ds = test_ds.rename_column(label_col, "labels")

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

columns_to_keep = ["input_ids", "attention_mask", "labels"]
if "id" in train_ds.column_names:
    columns_to_keep.append("id")

train_ds.set_format(type="torch", columns=[c for c in columns_to_keep if c in train_ds.column_names])
val_ds.set_format(type="torch", columns=[c for c in columns_to_keep if c in val_ds.column_names])
test_ds.set_format(type="torch", columns=[c for c in columns_to_keep if c in test_ds.column_names])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/29040 [00:00<?, ? examples/s]

Map:   0%|          | 0/6223 [00:00<?, ? examples/s]

Map:   0%|          | 0/6223 [00:00<?, ? examples/s]

Model + Training

In [ ]:

# 5. Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)


# 6. Training arguments
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    fp16=FP16,
    report_to="none",
    seed=SEED
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:

# 7. Train
train_result = trainer.train()


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

with open(OUTPUT_DIR / "train_result.json", "w") as f:
    json.dump(
        {
            "train_runtime": train_result.metrics.get("train_runtime"),
            "train_samples_per_second": train_result.metrics.get("train_samples_per_second"),
            "train_steps_per_second": train_result.metrics.get("train_steps_per_second"),
            "train_loss": train_result.metrics.get("train_loss"),
        },
        f,
        indent=2
    )

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.096574,0.487864,0.831432,0.824548
2,0.754665,0.488325,0.862767,0.856744
3,0.473330,0.598709,0.869677,0.862763
4,0.332893,0.786472,0.873534,0.865749


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# 8. Predict + save detailed outputs
def predict_and_save(
    trainer: Trainer,
    df: pd.DataFrame,
    hf_ds: Dataset,
    split_name: str,
    output_dir: Path,
    id2label: Dict[int, str]
) -> Tuple[pd.DataFrame, Dict]:
    pred_output = trainer.predict(hf_ds)
    logits = pred_output.predictions
    probs = softmax(logits)
    pred_ids = probs.argmax(axis=1)
    pred_labels = [id2label[int(i)] for i in pred_ids]
    true_ids = df[label_col].values
    true_labels = [id2label[int(i)] for i in true_ids]

    confidence = probs.max(axis=1)
    entropy = compute_entropy(probs)
    margin = compute_margin(probs)

    results_df = pd.DataFrame({
        "id": df[id_col].values,
        "sentence": df[text_col].values,
        "true_label_id": true_ids,
        "true_label": true_labels,
        "pred_label_id": pred_ids,
        "pred_label": pred_labels,
        "confidence": confidence,
        "entropy": entropy,
        "margin": margin
    })

    # Probability columns in canonical label order
    for class_name, class_id in label2id.items():
        results_df[f"prob_{class_name}"] = probs[:, class_id]

    results_df["correct"] = (results_df["true_label_id"] == results_df["pred_label_id"]).astype(int)

    # Metrics
    acc = accuracy_score(true_ids, pred_ids)
    macro_f1 = f1_score(true_ids, pred_ids, average="macro")
    cls_report = classification_report(true_ids, pred_ids, target_names=[id2label[i] for i in range(len(id2label))], output_dict=True)

    metrics = {
        "split": split_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "num_examples": int(len(results_df)),
        "classification_report": cls_report
    }

    results_df.to_csv(output_dir / f"{split_name}_predictions.csv", index=False)
    with open(output_dir / f"{split_name}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    save_confusion_matrix(
        true_ids,
        pred_ids,
        labels=[id2label[i] for i in range(len(id2label))],
        out_path=output_dir / f"{split_name}_confusion_matrix.png",
        title=f"RoBERTa-large Confusion Matrix ({split_name})"
    )

    return results_df, metrics


In [ ]:
val_results_df, val_metrics = predict_and_save(
    trainer, val_df, val_ds, "val", OUTPUT_DIR, id2label
)

test_results_df, test_metrics = predict_and_save(
    trainer, test_df, test_ds, "test", OUTPUT_DIR, id2label
)

print("Validation metrics:", val_metrics["accuracy"], val_metrics["macro_f1"])
print("Test metrics:")
print("Accuracy: ",test_metrics["accuracy"], "Macro F1", test_metrics["macro_f1"])


Validation metrics: 0.8735336654346778 0.8657489377852196
Test metrics:
Accuracy:  0.8764261610155873 Macro F1 0.8697122780510548


In [ ]:

# 9. Save run config
run_config = {
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "warmup_ratio": WARMUP_RATIO,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "seed": SEED,
    "train_path": str(TRAIN_PATH),
    "val_path": str(VAL_PATH),
    "test_path": str(TEST_PATH),
    "text_col": text_col,
    "label_col": label_col,
    "id_col": id_col,
    "label2id": label2id,
    "id2label": id2label
}

with open(OUTPUT_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)


In [ ]:

# 10. Merge predictions with linguistic features
# routing/error analysis.

# feature_cols = [
#     "token_count", "char_count", "avg_word_len", "dep_depth", "num_clauses",
#     "passive_voice_count", "negation_count", "hedge_count", "num_financial_terms",
#     "num_entities", "entity_density", "flesch_reading_ease", "flesch_kincaid_grade",
#     "complexity_score"
# ]

# existing_feature_cols = [c for c in feature_cols if c in val_df.columns]

# val_with_features = val_results_df.merge(
#     val_df[[id_col] + existing_feature_cols],
#     left_on="id",
#     right_on=id_col,
#     how="left"
# ).drop(columns=[id_col], errors="ignore")

# test_with_features = test_results_df.merge(
#     test_df[[id_col] + existing_feature_cols],
#     left_on="id",
#     right_on=id_col,
#     how="left"
# ).drop(columns=[id_col], errors="ignore")

# val_with_features.to_csv(OUTPUT_DIR / "val_predictions_with_features.csv", index=False)
# test_with_features.to_csv(OUTPUT_DIR / "test_predictions_with_features.csv", index=False)


In [ ]:
# 12. Summary CSV for report

# summary_df = pd.DataFrame([
#     {
#         "model": "roberta-large-finetuned",
#         "split": "val",
#         "accuracy": val_metrics["accuracy"],
#         "macro_f1": val_metrics["macro_f1"],
#         "num_examples": val_metrics["num_examples"]
#     },
#     {
#         "model": "roberta-large-finetuned",
#         "split": "test",
#         "accuracy": test_metrics["accuracy"],
#         "macro_f1": test_metrics["macro_f1"],
#         "num_examples": test_metrics["num_examples"]
#     }
# ])

# summary_df.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)
# print(summary_df)

# print(f"\nDone. Outputs saved to: {OUTPUT_DIR.resolve()}")
# print(f"Fine-tuned model saved to: {MODEL_DIR.resolve()}")

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Load predictions
df = pd.read_csv("/content/Linguistically-aware-model-routing-for-financial-sentiment-analysis/Output/model_outputs/Finetuned_Roberta_large/with_Combined_Dataset/test_predictions.csv") 

y_true = df["true_label"]
y_pred = df["pred_label"]

# Compute metrics
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("="*60)
print("Large Model (RoBERTa-large) — Test Set")
print("="*60)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")

print("\nPer-class report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "neutral", "positive"]
))

Large Model (RoBERTa-large) — Test Set

Accuracy: 0.8764
Macro-F1: 0.8697

Per-class report:

              precision    recall  f1-score   support

    negative       0.84      0.83      0.84      1343
     neutral       0.89      0.87      0.88      2143
    positive       0.89      0.90      0.89      2737

    accuracy                           0.88      6223
   macro avg       0.87      0.87      0.87      6223
weighted avg       0.88      0.88      0.88      6223

